# Hello Quant

In [5]:
# coding=utf-8
from __future__ import print_function, absolute_import, unicode_literals
import numpy as np
from gm.api import *
from pandas import DataFrame

'''
本策略每隔1个月定时触发,根据Fama-French三因子模型对每只股票进行回归，得到其alpha值。
假设Fama-French三因子模型可以完全解释市场，则alpha为负表明市场低估该股，因此应该买入。
策略思路：
计算市场收益率、个股的账面市值比和市值,并对后两个进行了分类,
根据分类得到的组合分别计算其市值加权收益率、SMB和HML. 
对各个股票进行回归(假设无风险收益率等于0)得到alpha值.
选取alpha值小于0并为最小的10只股票进入标的池
平掉不在标的池的股票并等权买入在标的池的股票
回测数据:SHSE.000300的成份股
回测时间:2017-07-01 08:00:00到2017-10-01 16:00:00
'''

def init(context):
    # 每月第一个交易日的09:40 定时执行algo任务（仿真和实盘时不支持该频率）
    schedule(schedule_func=algo, date_rule='1m', time_rule='09:40:00')

    # 数据滑窗
    context.date = 20

    # 设置开仓的最大资金量
    context.ratio = 0.8

    # 账面市值比的大/中/小分类
    context.BM_BIG = 3.0
    context.BM_MID = 2.0
    context.BM_SMA = 1.0

    # 市值大/小分类
    context.MV_BIG = 2.0
    context.MV_SMA = 1.0

# 计算市值加权的收益率的函数,MV为市值的分类对应的组别,BM为账目市值比的分类对应的组别
def market_value_weighted(stocks, MV, BM):
    select = stocks[(stocks['NEGOTIABLEMV'] == MV) & (stocks['BM'] == BM)] # 选出市值为MV，账目市值比为BM的所有股票数据
    market_value = select['mv'].values     # 对应组的全部市值数据
    mv_total = np.sum(market_value)        # 市值求和
    mv_weighted = [mv / mv_total for mv in market_value]   # 市值加权的权重
    stock_return = select['return'].values

    # 返回市值加权的收益率的和
    return_total = []
    for i in range(len(mv_weighted)):
        return_total.append(mv_weighted[i] * stock_return[i])
    return_total = np.sum(return_total)
    return return_total


def algo(context):
    # 获取上一个交易日的日期
    last_day = get_previous_trading_date(exchange='SHSE', date=context.now)

    # 获取沪深300成份股
    context.stock300 = get_history_constituents(index='SHSE.000300', start_date=last_day,
                                                end_date=last_day)[0]['constituents'].keys()

    # 获取当天有交易的股票
    not_suspended = get_history_instruments(symbols=context.stock300, start_date=last_day, end_date=last_day)
    not_suspended = [item['symbol'] for item in not_suspended
                     if not item['is_suspended'] and item['symbol']!= 'SHSE.601313']  # 601313 为退市股票
    fin = get_fundamentals(table='trading_derivative_indicator', symbols=not_suspended,
                           start_date=last_day, end_date=last_day,fields='PB,NEGOTIABLEMV', df=True)  # 获取P/B和市值数据

    # 计算账面市值比,为P/B的倒数
    fin['PB'] = (fin['PB'] ** -1)

    # 计算市值的50%的分位点,用于后面的分类
    size_gate = fin['NEGOTIABLEMV'].quantile(0.50)

    # 计算账面市值比的30%和70%分位点,用于后面的分类
    bm_gate = [fin['PB'].quantile(0.30), fin['PB'].quantile(0.70)]
    fin.index = fin.symbol

    # 设置存放股票收益率的list
    x_return = []

    # 对未停牌的股票进行处理
    for symbol in not_suspended:
        # 计算收益率，存放到x_return里面
        close = history_n(symbol=symbol, frequency='1d', count=context.date + 1, end_time=last_day, fields='close',
                          skip_suspended=True, fill_missing='Last', adjust=ADJUST_PREV, df=True)['close'].values
        stock_return = close[-1] / close[0] - 1
        pb = fin['PB'][symbol]
        market_value = fin['NEGOTIABLEMV'][symbol]

        # 获取[股票代码， 股票收益率, 账面市值比的分类, 市值的分类, 流通市值]
        # 其中账面市值比的分类为：大（3）、中（2）、小（1）
        # 流通市值的分类：大（2）、小（1）
        if pb < bm_gate[0]:
            if market_value < size_gate:
                label = [symbol, stock_return, context.BM_SMA, context.MV_SMA, market_value]
            else:
                label = [symbol, stock_return, context.BM_SMA, context.MV_BIG, market_value]
        elif pb < bm_gate[1]:
            if market_value < size_gate:
                label = [symbol, stock_return, context.BM_MID, context.MV_SMA, market_value]
            else:
                label = [symbol, stock_return, context.BM_MID, context.MV_BIG, market_value]
        elif market_value < size_gate:
            label = [symbol, stock_return, context.BM_BIG, context.MV_SMA, market_value]
        else:
            label = [symbol, stock_return, context.BM_BIG, context.MV_BIG, market_value]
        if len(x_return) == 0:
            x_return = label
        else:
            x_return = np.vstack([x_return, label])

    # 将股票代码、 股票收益率、 账面市值比的分类、 市值的分类、 流通市值存为数据表
    stocks = DataFrame(data=x_return, columns=['symbol', 'return', 'BM', 'NEGOTIABLEMV', 'mv'])
    stocks.index = stocks.symbol
    columns = ['return', 'BM', 'NEGOTIABLEMV', 'mv']

    for column in columns:
        stocks[column] = stocks[column].astype(np.float64)

    # 计算SMB.HML和市场收益率（市值加权法）
    smb_s = (market_value_weighted(stocks, context.MV_SMA, context.BM_SMA) +
             market_value_weighted(stocks, context.MV_SMA, context.BM_MID) +
             market_value_weighted(stocks, context.MV_SMA, context.BM_BIG)) / 3

    # 获取大市值组合的市值加权组合收益率
    smb_b = (market_value_weighted(stocks, context.MV_BIG, context.BM_SMA) +
             market_value_weighted(stocks, context.MV_BIG, context.BM_MID) +
             market_value_weighted(stocks, context.MV_BIG, context.BM_BIG)) / 3
    smb = smb_s - smb_b

    # 获取大账面市值比组合的市值加权组合收益率
    hml_b = (market_value_weighted(stocks, context.MV_SMA, 3) +
             market_value_weighted(stocks, context.MV_BIG, context.BM_BIG)) / 2

    # 获取小账面市值比组合的市值加权组合收益率
    hml_s = (market_value_weighted(stocks, context.MV_SMA, context.BM_SMA) +
             market_value_weighted(stocks, context.MV_BIG, context.BM_SMA)) / 2
    hml = hml_b - hml_s

    # 获取市场收益率
    close = history_n(symbol='SHSE.000300', frequency='1d', count=context.date + 1,
                      end_time=last_day, fields='close', skip_suspended=True,
                      fill_missing='Last', adjust=ADJUST_PREV, df=True)['close'].values
    market_return = close[-1] / close[0] - 1
    coff_pool = []

    # 对每只股票进行回归获取其alpha值
    for stock in stocks.index:
        x_value = np.array([[market_return], [smb], [hml], [1.0]])
        y_value = np.array([stocks['return'][stock]])
        # OLS估计系数
        coff = np.linalg.lstsq(x_value.T, y_value)[0][3]
        coff_pool.append(coff)

    # 获取alpha最小并且小于0的10只的股票进行操作(若少于10只则全部买入)
    stocks['alpha'] = coff_pool
    stocks = stocks[stocks.alpha < 0].sort_values(by='alpha').head(10)
    symbols_pool = stocks.index.tolist()
    positions = context.account().positions()

    # 平不在标的池的股票
    for position in positions:
        symbol = position['symbol']
        if symbol not in symbols_pool:
            order_target_percent(symbol=symbol, percent=0, order_type=OrderType_Market,
                                 position_side=PositionSide_Long)
            print('市价单平不在标的池的', symbol)

    # 获取股票的权重
    percent = context.ratio / len(symbols_pool)

    # 买在标的池中的股票
    for symbol in symbols_pool:
        order_target_percent(symbol=symbol, percent=percent, order_type=OrderType_Market,
                             position_side=PositionSide_Long)
        print(symbol, '以市价单调多仓到仓位', percent)


def main():
    '''
    strategy_id策略ID,由系统生成
    filename文件名,请与本文件名保持一致
    mode实时模式:MODE_LIVE回测模式:MODE_BACKTEST
    token绑定计算机的ID,可在系统设置-密钥管理中生成
    backtest_start_time回测开始时间
    backtest_end_time回测结束时间
    backtest_adjust股票复权方式不复权:ADJUST_NONE前复权:ADJUST_PREV后复权:ADJUST_POST
    backtest_initial_cash回测初始资金
    backtest_commission_ratio回测佣金比例
    backtest_slippage_ratio回测滑点比例
    '''
    run(strategy_id='strategy_id',
        filename='1th.py',
        mode=MODE_BACKTEST,
        token='ee80a887e88e31d6b9a4b0d528f80f9037a9fac0',
        backtest_start_time='2017-07-01 08:00:00',
        backtest_end_time='2017-10-01 16:00:00',
        backtest_adjust=ADJUST_PREV,
        backtest_initial_cash=10000000,
        backtest_commission_ratio=0.0001,
        backtest_slippage_ratio=0.0001)

main()

Usage: ipykernel_launcher.py [options]

ipykernel_launcher.py: error: no such option: -f
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/lib/python3.8/optparse.py", line 1387, in parse_args
    stop = self._process_args(largs, rargs, values)
  File "/usr/lib/python3.8/optparse.py", line 1431, in _process_args
    self._process_short_opts(rargs, values)
  File "/usr/lib/python3.8/optparse.py", line 1513, in _process_short_opts
    raise BadOptionError(opt)
optparse.BadOptionError: no such option: -f

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/py38/lib/python3.8/site-packages/IPython/core/interactiveshell.py", line 3437, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "<ipython-input-5-d0377338cb2f>", line 205, in <module>
    main()
  File "<ipython-input-5-d0377338cb2f>", line 194, in main
    run(strategy_id='strategy_id',
  File "/opt/py38/lib/python3.8/site-packages/gm/api/basic.py", line 336, in run
    (options, args) = parser.parse_args()
  File "/usr/lib/python3.8/optp

TypeError: object of type 'NoneType' has no len()

In [3]:
#from .debug import *

from gm.api import *
from debug import launcher

set_token('ee80a887e88e31d6b9a4b0d528f80f9037a9fac0')



# 查询历史行情, 采用定点复权的方式， adjust指定前复权，adjust_end_time指定复权时间点
data = history(symbol='SHSE.600000', frequency='1d', start_time='2020-01-01 09:00:00', end_time='2020-12-31 16:00:00',
               fields='open,high,low,close', adjust=ADJUST_PREV, adjust_end_time='2020-12-31', df=True)
print(data)



2021-12-13 19:16:46,070 WARNING [history.py:26] error code=1001, error info=无法连接到终端服务


2021-12-13 19:16:46,074 WARNING [history.py:26] error code=1001, error info=无法连接到终端服务


2021-12-13 19:16:46,077 WARNING [history.py:26] error code=1001, error info=无法连接到终端服务


2021-12-13 19:16:46,078 ERROR [query.py:99] 获取不到数据服务地址
Traceback (most recent call last):
  File "/opt/py38/lib/python3.8/site-packages/gm/api/query.py", line 52, in wrapper
    return func(*args, **kw)
  File "/opt/py38/lib/python3.8/site-packages/gm/retrying.py", line 37, in wrapped_f
    return Retrying(*dargs, **dkw).call(f, *args, **kw)
  File "/opt/py38/lib/python3.8/site-packages/gm/retrying.py", line 202, in call
    raise attempt.get()
  File "/opt/py38/lib/python3.8/site-packages/gm/retrying.py", line 240, in get
    six.reraise(self.value[0], self.value[1], self.value[2])
  File "/opt/py38/lib/python3.8/site-packages/six.py", line 703, in reraise
    raise value
  File "/opt/py38/lib/python3.8/site-packages/gm/retrying.py", line 190, in call
    attempt = Attempt(fn(*args, **kwargs), attempt_number, False)
  File "/opt/py38/lib/python3.8/site-packages/gm/api/query.py", line 372, in history
    return historyapi.get_history_bars(symbols=symbol, frequency=frequency,
  File "/o

ERROR:gmsdklogger:获取不到数据服务地址
Traceback (most recent call last):
  File "/opt/py38/lib/python3.8/site-packages/gm/api/query.py", line 52, in wrapper
    return func(*args, **kw)
  File "/opt/py38/lib/python3.8/site-packages/gm/retrying.py", line 37, in wrapped_f
    return Retrying(*dargs, **dkw).call(f, *args, **kw)
  File "/opt/py38/lib/python3.8/site-packages/gm/retrying.py", line 202, in call
    raise attempt.get()
  File "/opt/py38/lib/python3.8/site-packages/gm/retrying.py", line 240, in get
    six.reraise(self.value[0], self.value[1], self.value[2])
  File "/opt/py38/lib/python3.8/site-packages/six.py", line 703, in reraise
    raise value
  File "/opt/py38/lib/python3.8/site-packages/gm/retrying.py", line 190, in call
    attempt = Attempt(fn(*args, **kwargs), attempt_number, False)
  File "/opt/py38/lib/python3.8/site-packages/gm/api/query.py", line 372, in history
    return historyapi.get_history_bars(symbols=symbol, frequency=frequency,
  File "/opt/py38/lib/python3.8/site

Empty DataFrame
Columns: []
Index: []
